# 1 — Source categories: the foundation of retrieval steering

The EMA corpus is dominated by **product-specific** documents (~18k EPAR assessment
reports among 79,882 docs), so a plain vector top-k often comes back EPAR-saturated even
when the question asks about *general* requirements that live in scientific guidelines or
EMA Q&A pages. The steering stack (see [`docs/RETRIEVAL.md`](../RETRIEVAL.md) §7) counters
that with three composable mechanisms:

| | Mechanism | Acts at | Notebook |
|---|---|---|---|
| **A** | category **filter** (per call) + **quota** (per profile) | candidate stage | [02](02_steered_retrieval.ipynb) |
| **B** | **`LINKS_TO` expansion** toward configured categories | expansion stage | [02](02_steered_retrieval.ipynb) |
| **C** | query→category **routing table** | before retrieval | [03](03_routing_and_full_agent.ipynb) |

All three rest on one shared foundation — a **source category** per document — which is
what this notebook covers:

1. the category vocabulary and the pure classifier,
2. the persisted `:Document.category` property in Neo4j,
3. the one-off backfill for an existing graph.

**Prerequisites:** the data services are up (`scripts/start_services.sh`) and
`~/.myenvs/ema_nlp.env` holds the Neo4j credentials. Everything in section 1 also runs
fully offline.

In [ ]:
# Environment: credentials live in ~/.myenvs/ema_nlp.env (never in the repo).
# On this host the project Neo4j container runs on alt ports; if your env file
# does not set NEO4J_URI, the setdefault below points at the project container.
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

load_dotenv(Path.home() / ".myenvs" / "ema_nlp.env")
os.environ.setdefault("NEO4J_URI", "bolt://localhost:7688")  # project container (alt port)

# Make `harness` importable when the notebook is started from docs/examples/.
REPO = Path.cwd()
while not (REPO / "harness").exists() and REPO != REPO.parent:
    REPO = REPO.parent
sys.path.insert(0, str(REPO))
os.chdir(REPO)
print("repo root:", REPO)

## 1. The category vocabulary

`harness/retrieval/doc_categories.py` defines a small, ordered vocabulary and a pure,
table-driven classifier over a document's URL + topic path. Nothing downstream hardcodes
a category — filters, quotas, expansion targets, and routing rules all take values from
this vocabulary via config or tool arguments, so extending it is a data change plus a
rule entry, not a code change.

In [ ]:
from harness.retrieval.doc_categories import CATEGORIES, classify_source

print("vocabulary (most-authoritative-first):", list(CATEGORIES))

examples = [
    "https://www.ema.europa.eu/en/documents/scientific-guideline/ich-q3d-elemental-impurities_en.pdf",
    "https://www.ema.europa.eu/en/documents/other/questions-answers-nitrosamine-impurities_en.pdf",
    "https://www.ema.europa.eu/en/documents/assessment-report/keytruda-epar-public-assessment-report_en.pdf",
    "https://www.ema.europa.eu/en/medicines/human/EPAR/keytruda",
    "https://www.ema.europa.eu/en/about-us/contact",
]
for url in examples:
    print(f"{classify_source(url):22s} <- {url}")

## 2. The persisted `:Document.category` property

Classification at read time is enough to *display* a category, but Cypher-side
**filtering** (Option A) and **expansion targeting** (Option B) need the category as a
node property inside Neo4j. New index builds stamp it at ingest
(`harness/indexing/property_graph.py::_entity_for`); an existing graph gets it from the
backfill script below.

First, check the current state of your graph:

In [ ]:
from harness.indexing.property_graph import neo4j_store_from_env

store = neo4j_store_from_env()
rows = store.structured_query(
    "MATCH (d:Document) "
    "RETURN coalesce(d.category, '(no category - backfill needed)') AS category, "
    "count(*) AS docs ORDER BY docs DESC"
)
for r in rows:
    print(f"{r['category']:35s} {r['docs']}")

## 3. The backfill (one-off, idempotent)

`scripts/backfill_doc_categories.py` classifies every `:Document` with the same
table-driven rules as the runtime classifier — so the stored property and the derived
value can never disagree — and writes `d.category` in batches. It touches **only** that
property: chunks, embeddings, and edges are untouched, and it is safe to re-run any time
(e.g. after extending the classification rules).

Always look at the `--dry-run` histogram first; on the full graph you would expect a
large `epar` + `medicine_page` share and a much smaller `scientific_guideline` + `qa`
share — that skew is exactly why steering exists.

In [ ]:
!python scripts/backfill_doc_categories.py --dry-run

In [ ]:
# When the histogram looks sane, write it (idempotent, property-only):
!python scripts/backfill_doc_categories.py

## Where you are now

The graph carries a filterable `category` on every document — the substrate the next two
notebooks build on:

- **[02_steered_retrieval.ipynb](02_steered_retrieval.ipynb)** — steer the retriever
  itself: per-call filters, per-profile quotas (Option A) and link-graph expansion
  (Option B).
- **[03_routing_and_full_agent.ipynb](03_routing_and_full_agent.ipynb)** — routing tables
  (Option C) and the complete headless pipeline (tool → agent → structured answer).